# Pix2Pix Training — v3

Trains two models from scratch on the pooled v3 dataset:
- **HE_PAS_model** : HE → PAS  (samples 1–4, 8)
- **HE_TRI_model** : HE → TRI  (samples 1–8)

Data is in `data_v3/HE_PAS/AB/` and `data_v3/HE_TRI/AB/` — the side-by-side format expected by pix2pix `--dataset_mode aligned`.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
REPO      = r'' # path to pix2pix model
DATA_ROOT = r'' # path to data

# ── Training hyperparameters ───────────────────────────────────────────────────
N_EPOCHS        = 100   # epochs with constant learning rate
N_EPOCHS_DECAY  = 100   # epochs to linearly decay lr to 0 (total = 200 epochs)
BATCH_SIZE      = 1     # pix2pix default; increase if GPU memory allows
LOAD_SIZE       = 286   # image is first resized to this, then randomly cropped
CROP_SIZE       = 256   # final training crop size (pix2pix default)
DIRECTION       = 'AtoB'  # A = HE, B = PAS or TRI
GPU_ID          = 0

In [ ]:
%cd "" # path to change directory to

## 1. Check GPU

In [ ]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(GPU_ID)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(GPU_ID).total_memory / 1e9:.1f} GB")

## 2. Check Dataset

In [ ]:
import os

for dataset in ('HE_PAS', 'HE_TRI'):
    print(f"\n{dataset}")
    for split in ('train', 'val', 'test'):
        split_dir = os.path.join(DATA_ROOT, dataset, 'AB', split)
        if os.path.exists(split_dir):
            n = len(os.listdir(split_dir))
            print(f"  {split:6s} : {n:5d} images")
        else:
            print(f"  {split:6s} : NOT FOUND — run the crop notebook first")

In [ ]:
# Start this in a separate terminal BEFORE running train.py
!python -m visdom.server

## 3. Train HE → PAS

In [ ]:
!python "{REPO}/train.py" \
    --dataroot "{DATA_ROOT}/HE_PAS/AB" \
    --name HE_PAS_model_v3 \
    --model pix2pix \
    --direction {DIRECTION} \
    --dataset_mode aligned \
    --n_epochs {N_EPOCHS} \
    --n_epochs_decay {N_EPOCHS_DECAY} \
    --batch_size {BATCH_SIZE} \
    --load_size {LOAD_SIZE} \
    --crop_size {CROP_SIZE} \
    --save_latest_freq 5000 \
    --save_epoch_freq 5 \
    --verbose

## 4. Train HE → TRI

In [ ]:
!python "{REPO}/train.py" \
    --dataroot "{DATA_ROOT}/HE_TRI/AB" \
    --name HE_TRI_model_v3 \
    --model pix2pix \
    --direction {DIRECTION} \
    --dataset_mode aligned \
    --n_epochs {N_EPOCHS} \
    --n_epochs_decay {N_EPOCHS_DECAY} \
    --batch_size {BATCH_SIZE} \
    --load_size {LOAD_SIZE} \
    --crop_size {CROP_SIZE} \
    --save_latest_freq 5000 \
    --save_epoch_freq 5 \
    --verbose

## 5. Plot Training Loss

Run this after training to visualise G and D losses over epochs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_loss(model_name):
    loss_path = os.path.join(REPO, 'checkpoints', model_name, 'loss_log.txt')
    if not os.path.exists(loss_path):
        print(f"Loss log not found: {loss_path}")
        return

    epochs, G_GAN, G_L1, D_real, D_fake = [], [], [], [], []

    with open(loss_path) as f:
        for line in f:
            if not line.startswith('(epoch'): continue
            parts = dict(p.strip().split(': ') for p in line.split(')')[1].split(',') if ': ' in p)
            ep = int(line.split('epoch: ')[1].split(',')[0])
            epochs.append(ep)
            G_GAN.append(float(parts.get('G_GAN', 0)))
            G_L1.append(float(parts.get('G_L1', 0)))
            D_real.append(float(parts.get('D_real', 0)))
            D_fake.append(float(parts.get('D_fake', 0)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'Training loss — {model_name}', fontsize=12)

    axes[0].plot(epochs, G_GAN, label='G_GAN')
    axes[0].plot(epochs, G_L1,  label='G_L1')
    axes[0].set_title('Generator')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(epochs, D_real, label='D_real')
    axes[1].plot(epochs, D_fake, label='D_fake')
    axes[1].set_title('Discriminator')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_loss('HE_PAS_model_v3')
plot_loss('HE_TRI_model_v3')

## 6. Quick Visual Check on Validation Set

Run inference on the val split and display a few real vs fake comparisons.

In [ ]:
import cv2

def quick_val_check(model_name, dataset_name, n_show=6):

    # Run inference on val set
    !python "{REPO}/test.py" \
        --dataroot "{DATA_ROOT}/{dataset_name}/AB" \
        --name {model_name} \
        --model pix2pix \
        --direction {DIRECTION} \
        --dataset_mode aligned \
        --phase val \
        --num_test {n_show} \
        --gpu_ids {GPU_ID} \
        --display_id 0

    results_dir = os.path.join(REPO, 'results', model_name, 'test_latest', 'images')
    if not os.path.exists(results_dir):
        print("Results not found — check that test.py ran successfully")
        return

    fake_files = sorted([f for f in os.listdir(results_dir) if 'fake' in f])[:n_show]

    fig, axes = plt.subplots(n_show, 3, figsize=(12, n_show * 4))
    fig.suptitle(f'{model_name} — val set: Real A | Fake B | Real B', fontsize=12)

    for row, fname in enumerate(fake_files):
        base = fname.replace('_fake_B', '')
        real_a = cv2.cvtColor(cv2.imread(os.path.join(results_dir, base + '_real_A.png')), cv2.COLOR_BGR2RGB)
        fake_b = cv2.cvtColor(cv2.imread(os.path.join(results_dir, fname)),                cv2.COLOR_BGR2RGB)
        real_b = cv2.cvtColor(cv2.imread(os.path.join(results_dir, base + '_real_B.png')), cv2.COLOR_BGR2RGB)

        for col, (title, img) in enumerate([('Real A (HE)', real_a),
                                             ('Fake B (generated)', fake_b),
                                             ('Real B (target)', real_b)]):
            axes[row, col].imshow(img)
            if row == 0:
                axes[row, col].set_title(title, fontsize=11)
            axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()


quick_val_check('HE_PAS_model_v3', 'HE_PAS')
quick_val_check('HE_TRI_model_v3', 'HE_TRI')